In [0]:
import re
import requests
from pathlib import Path

BASE_URL = "https://download.bls.gov/pub/time.series/pr/"
VOLUME_DIR = "/Volumes/dmp/rearc/quest/bls/pr"  # <-- edit this
HEADERS = {"User-Agent": "yourcompany-dataeng contact@yourcompany.com"}

Path(VOLUME_DIR).mkdir(parents=True, exist_ok=True)

# 1. Get the directory listing and pull out filenames
resp = requests.get(BASE_URL, headers=HEADERS, timeout=30)
resp.raise_for_status()

# BLS's index page lists filenames as plain text after the timestamp/size columns
filenames = re.findall(r'href="([^"?/][^"]*)"', resp.text)
if not filenames:
    # fallback: BLS sometimes serves a plain-text-style listing, not real <a href>
    filenames = re.findall(r'\bpr\.\S+', resp.text)

filenames = sorted(set(filenames))
print(f"Found {len(filenames)} files: {filenames}")

# 2. Download each file and land it raw in the Volume
for fname in filenames:
    url = BASE_URL + fname
    r = requests.get(url, headers=HEADERS, timeout=60)
    if r.status_code != 200:
        print(f"FAILED ({r.status_code}): {fname}")
        continue
    out_path = Path(VOLUME_DIR) / fname
    out_path.write_bytes(r.content)
    print(f"Saved {fname} ({len(r.content)} bytes) -> {out_path}")

print("Done.")

In [0]:
import re
import requests
from pathlib import Path

DOMAIN = "https://download.bls.gov"
BASE_URL = f"{DOMAIN}/pub/time.series/pr/"
VOLUME_DIR = "/Volumes/dmp/rearc/quest/bls/pr"
HEADERS = {"User-Agent": "yourcompany-dataeng contact@yourcompany.com"}

Path(VOLUME_DIR).mkdir(parents=True, exist_ok=True)

resp = requests.get(BASE_URL, headers=HEADERS, timeout=30)
resp.raise_for_status()

hrefs = re.findall(r'<A HREF="([^"]+)">', resp.text, flags=re.IGNORECASE)
file_paths = [h for h in hrefs if "/pr/pr." in h]

print(f"Found {len(file_paths)} files:")
for p in file_paths:
    print(" ", p)

for path in file_paths:
    fname = path.rsplit("/", 1)[-1]
    url = DOMAIN + path
    try:
        r = requests.get(url, headers=HEADERS, timeout=60)
    except requests.exceptions.RequestException as e:
        print(f"SKIPPED (error {e}): {fname}")
        continue

    if r.status_code != 200:
        print(f"FAILED ({r.status_code}): {fname}")
        continue

    out_path = Path(VOLUME_DIR) / fname
    out_path.write_bytes(r.content)
    print(f"Saved {fname} ({len(r.content)} bytes) -> {out_path}")

print("Done.")

In [0]:
import json
import requests
from pathlib import Path

VOLUME_DIR = "/Volumes/dmp/rearc/quest/bls/pr"  # or a separate folder if you prefer
POP_API_URL = "https://honolulu-api.datausa.io/tesseract/data.jsonrecords?cube=acs_yg_total_population_1&drilldowns=Year%2CNation&locale=en&measures=Population"
HEADERS = {"User-Agent": "yourcompany-dataeng contact@yourcompany.com"}

Path(VOLUME_DIR).mkdir(parents=True, exist_ok=True)

resp = requests.get(POP_API_URL, headers=HEADERS, timeout=60)
resp.raise_for_status()

data = resp.json()  # validates it's actually JSON before writing

out_path = Path(VOLUME_DIR) / "population_acs_yg_total_population_1.json"
out_path.write_text(json.dumps(data, indent=2))

print(f"Saved population data ({len(resp.content)} bytes) -> {out_path}")